In [1]:
import torch
from torchvision import datasets

In [2]:
train_dataset = datasets.MNIST(
    root = "data",
    train = True,
    download = True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.68MB/s]


In [3]:
print(len(train_dataset))

60000


In [4]:
image, label = train_dataset[0]

print(type(image))
print(label)
print(image.size)

<class 'PIL.Image.Image'>
5
(28, 28)


In [5]:
first_sample = train_dataset[0]

print(type(first_sample))

<class 'tuple'>


In [6]:
print(type(first_sample[0]))
print(type(first_sample[1]))

<class 'PIL.Image.Image'>
<class 'int'>


In [7]:
print(image.size)

(28, 28)


In [8]:
image.show()

In [9]:
import numpy as np

image_array = np.array(image)

In [10]:
type(image_array)

numpy.ndarray

In [11]:
image_array.shape

(28, 28)

In [12]:
image_array[14][14]

np.uint8(240)

In [13]:
from torchvision import transforms

In [14]:
transform = transforms.ToTensor()

In [15]:
print(type(transform))

<class 'torchvision.transforms.transforms.ToTensor'>


In [16]:
train_dataset = datasets.MNIST(
    root = "data",
    train = True,
    download = True,
    transform = transforms.ToTensor()
)

In [17]:
first_sample = train_dataset[0]

In [18]:
type(first_sample[0])

torch.Tensor

In [19]:
first_sample[0].shape

torch.Size([1, 28, 28])

In [20]:
from torch.utils.data import DataLoader

In [21]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64
)

In [22]:
iterator = iter(train_loader)
first_batch = next(iterator)

In [23]:
type(first_batch)

list

In [24]:
print(type(first_batch))
print(len(first_batch))

<class 'list'>
2


In [25]:
print(type(first_batch[0]))
print(type(first_batch[1]))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [26]:
print(first_batch[0].shape)
print(first_batch[1].shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [27]:
import torch.nn as nn

class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(784,128)
    self.layer2 = nn.Linear(128,64)
    self.layer3 = nn.Linear(64,10)

    self.relu = nn.ReLU()

  def forward(self, x):
   x = torch.flatten(x, start_dim=1)

   x = self.layer1(x)

   x = self.relu(x)

   x = self.layer2(x)

   x = self.relu(x)

   x = self.layer3(x)

   return x

In [28]:
model = NeuralNetwork()

loss_fn = nn.CrossEntropyLoss()

In [29]:
output = model(first_batch[0])

In [30]:
print(output.shape)

torch.Size([64, 10])


In [31]:
labels = first_batch[1]
loss = loss_fn(output, labels)
print(loss)

tensor(2.2977, grad_fn=<NllLossBackward0>)


In [32]:
import torch.optim as optim

In [33]:
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

In [35]:
outputs = model(first_batch[0])

labels = first_batch[1]

loss = loss_fn(outputs, labels)

optimizer.zero_grad()

loss.backward()

optimizer.step()

In [36]:
epochs = 5

for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.3491
Epoch 2/5, Loss: 0.1519
Epoch 3/5, Loss: 0.1046
Epoch 4/5, Loss: 0.0777
Epoch 5/5, Loss: 0.0591


In [37]:
test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [38]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.93%


In [40]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params}")

Total Parameters: 109386
